# Study Assistant — Orbit

**Lumexa AI Builder course**

A tutoring companion, **Orbit**, that helps you learn by guiding your
thinking with hints and questions — instead of just handing over answers.

This notebook is fully self-contained and works with **Runtime → Run all** —
no setup, no API keys required, and nothing to upload.

**What you'll do:**
1. Build Orbit's "guide, don't just give answers" tutoring system prompt
   (reused verbatim from the original Streamlit app's persona rules).
2. Set up an explicit, inspectable session state (subject + level).
3. Generate a practice question and watch a scripted demo prove the
   hint-first pedagogy: a wrong attempt gets a hint, not the answer; asking
   directly for the answer gets the answer.
4. Learn about an *optional* upgrade path: paste in a real OpenAI API key to
   let Orbit tutor using `gpt-4o-mini` — completely optional.

**Why no required API key / no downloaded AI model?** So this notebook
always finishes `Runtime → Run all` for every student, instantly, with zero
setup friction. The optional OpenAI section shows what a real LLM tutor
sounds like.


## Step 1: Orbit's tutoring persona and pedagogy rules

`build_system_prompt(subject, level)` is reused verbatim from the original
`src/app.py` Streamlit app. It's used two ways here:

- As the literal system prompt if you opt in to the real OpenAI API later.
- As documentation of the pedagogy rules our rule-based tutoring engine
  below tries to follow (hint before answer, encouraging tone, on-topic).


In [1]:
def build_system_prompt(subject: str, level: str) -> str:
    """Build Orbit's system prompt, tailored to the chosen subject/level.

    The prompt enforces a 'guide, don't just give answers' tutoring style
    as an explicit behavioral rule, not just a tone instruction.
    """
    return f"""
You are Orbit, a friendly AI study companion aboard the Lumexa space station.

PERSONA:
- Warm, encouraging tone, like a patient older sibling.
- At most one light space metaphor per response.
- Keep responses concise: 3-5 sentences unless asked for more detail.

CURRENT SESSION:
- Subject: {subject}
- Explanation level: {level}
- Tailor vocabulary, pacing, and depth of explanation to this level.

PURPOSE (VERY IMPORTANT — TUTORING RULES):
- Your job is to help the student understand {subject} by guiding their
  thinking, NOT by doing their work for them.
- When the student asks a question, first respond with a guiding question
  or a helpful hint, rather than the direct final answer.
- Only give a direct, complete answer if the student has clearly made a
  genuine attempt AND explicitly asks for the answer (e.g., "just tell me
  the answer" or "I've tried and I'm stuck, please explain it").
- If asked, you can generate a practice question at the current subject
  and level. After the student answers, tell them if they're right, and
  if not, guide them toward the correct answer with hints rather than
  immediately stating it.

BOUNDARIES:
- Never write full graded assignments or essays for the student.
- If asked about something unrelated to schoolwork or study skills, gently
  redirect: "That's outside my mission here in the study lab — let's get
  back to your studies!"
- If you're not confident about a fact, say so honestly rather than guessing.

Stay in character as Orbit for the entire conversation.
""".strip()


print(build_system_prompt("Math", "Beginner")[:200] + "...")


You are Orbit, a friendly AI study companion aboard the Lumexa space station.

PERSONA:
- Warm, encouraging tone, like a patient older sibling.
- At most one light space metaphor per response.
- Keep ...


## Step 2: Explicit session state and conversation memory

Just like the recipe chatbot, `conversation_history` is the growing memory
list resent on every (optional) OpenAI call. `session_state` is a small,
explicit dictionary tracking the current subject/level and whichever
practice question is active — concrete state you can print and reason
about, mirroring what `st.session_state` did in the original Streamlit app.


In [2]:
SUBJECTS = ["Math", "Science", "English", "History", "Coding"]
LEVELS = ["Beginner", "Intermediate", "Advanced"]

session_state = {
    "subject": "Math",
    "level": "Beginner",
    "active_question": None,   # (question, hint, answer) tuple once a practice question is asked
}

conversation_history = [
    {"role": "system", "content": build_system_prompt(session_state["subject"], session_state["level"])}
]

print("session_state:", session_state)


session_state: {'subject': 'Math', 'level': 'Beginner', 'active_question': None}


## Step 3: A small practice-question bank

Each entry is a `(question, hint, answer)` tuple. The rule-based engine
picks one, gives a hint (not the answer) on a first attempt, checks a
correct attempt by simple text matching, and only reveals the answer if the
student explicitly asks for it or gets it right.


In [3]:
PRACTICE_BANK = {
    "Math": {
        "Beginner": [
            ("What is 7 + 8?", "Try breaking 8 into 3 + 5, so it's 7 + 3 + 5.", "15"),
            ("What is 9 x 6?", "Think of it as 9 x 5 plus one more 9.", "54"),
            ("What is 100 - 37?", "Try counting up from 37 to 100 in two steps: to 40, then to 100.", "63"),
        ],
        "Intermediate": [
            ("Solve for x: 2x + 5 = 17", "First isolate the term with x by subtracting 5 from both sides.", "6"),
            ("What is the area of a rectangle 6cm by 9cm?", "Area of a rectangle is length times width.", "54"),
        ],
        "Advanced": [
            ("Solve for x: x^2 - 5x + 6 = 0", "Try factoring into two binomials that multiply to 6 and add to -5.", "2 or 3"),
        ],
    },
    "Science": {
        "Beginner": [
            ("What gas do plants absorb from the air to make food?", "Think about what humans breathe out and plants take in.", "carbon dioxide"),
            ("What is the closest planet to the Sun?", "It's a small, rocky planet, first in line.", "mercury"),
        ],
        "Intermediate": [
            ("What is the powerhouse of the cell?", "It's an organelle known for producing energy (ATP).", "mitochondria"),
        ],
        "Advanced": [
            ("What force keeps planets in orbit around the Sun?", "Think about what Newton described with falling apples.", "gravity"),
        ],
    },
    "English": {
        "Beginner": [
            ("What do we call a word that describes a noun?", "It usually answers 'what kind' or 'how many'.", "adjective"),
        ],
        "Intermediate": [
            ("What literary device compares two things using 'like' or 'as'?", "It's different from a metaphor because of those connecting words.", "simile"),
        ],
        "Advanced": [
            ("What is it called when a story is told out of chronological order?", "Think about flashbacks and how they reorder events.", "nonlinear narrative"),
        ],
    },
    "History": {
        "Beginner": [
            ("In what year did World War II end?", "It ended in the mid-1940s, a few years after it began in 1939.", "1945"),
        ],
        "Intermediate": [
            ("What ancient civilization built the pyramids at Giza?", "Think of the Nile river and hieroglyphics.", "egyptians"),
        ],
        "Advanced": [
            ("What treaty officially ended World War I?", "It was signed at a palace outside Paris in 1919.", "treaty of versailles"),
        ],
    },
    "Coding": {
        "Beginner": [
            ("In Python, what keyword starts a function definition?", "It's a three-letter word, short for 'define'.", "def"),
        ],
        "Intermediate": [
            ("What data structure uses key-value pairs in Python?", "It's written with curly braces, like {'a': 1}.", "dictionary"),
        ],
        "Advanced": [
            ("What's the time complexity of binary search on a sorted list?", "Think about how the search space halves each step.", "o(log n)"),
        ],
    },
}


def get_practice_question(subject, level, index=None):
    """Pick a practice question, store it as the active question, and return the prompt text.

    Pass an explicit `index` for a reproducible pick (used by the scripted
    demo below); leave it as None for a pseudo-random pick (used when you
    try it yourself)."""
    bank = PRACTICE_BANK.get(subject, PRACTICE_BANK["Math"]).get(level, PRACTICE_BANK[subject]["Beginner"])
    question, hint, answer = bank[index] if index is not None else rng.choice(bank)
    session_state["active_question"] = {"question": question, "hint": hint, "answer": answer}
    return question


## Step 4: The rule-based tutoring engine (default path, no key needed)

`generate_rule_based_reply()` implements the hint-first pedagogy directly
in code:

- If the student explicitly asks for the answer ("just tell me", "I give
  up", "I'm stuck", ...), give the direct answer.
- If a practice question is active and the student's message contains the
  correct answer text, praise them.
- Otherwise, if a practice question is active, give the hint instead of the
  answer — matching "guide, don't just give answers".
- If nothing is active, respond with a generic guiding nudge rather than
  trying to actually solve an arbitrary open-ended question (that's the
  honest limitation of a rule-based engine — see the note after the demo).


In [4]:
import random
import re

RNG_SEED = 3
rng = random.Random(RNG_SEED)


def has_phrase(text, phrase):
    pattern = r"\b" + re.escape(phrase) + r"\b"
    return re.search(pattern, text) is not None


ASK_FOR_ANSWER_PHRASES = [
    "just tell me", "give me the answer", "i give up", "i'm stuck", "im stuck",
    "tell me the answer", "what's the answer", "whats the answer", "i don't know", "i dont know",
]
OFF_TOPIC_KEYWORDS = ["weather", "movie", "celebrity", "video game", "sports score"]
STUDY_HINT_KEYWORDS = ["study", "homework", "question", "answer", "subject", "learn"]

ENCOURAGEMENT_PHRASES = [
    "Nice thinking so far!",
    "You're on the right track, cadet.",
    "Good effort!",
]
HINT_TEMPLATES = [
    "Not quite yet — here's a hint: {hint}",
    "Close! Let's use a hint instead of the answer: {hint}",
    "Good attempt! Try this hint: {hint}",
]
CORRECT_TEMPLATES = [
    "That's correct! Great work — the answer is {answer}.",
    "You got it! {answer} is right.",
]
DIRECT_ANSWER_TEMPLATES = [
    "Since you've given it a genuine shot, here's the answer: {answer}.",
    "No problem, here's the direct answer: {answer}.",
]
GENERIC_GUIDE_TEMPLATES = [
    "Good question! Before I answer, what part of {subject} have you already tried or thought about?",
    "Let's launch into that together — what do you already know that might help with this {subject} question?",
]
BOUNDARY_REPLY = "That's outside my mission here in the study lab — let's get back to your studies!"


def generate_rule_based_reply(user_message, session_state, rng):
    lowered = user_message.lower()
    subject = session_state["subject"]
    active = session_state["active_question"]

    if any(has_phrase(lowered, kw) for kw in OFF_TOPIC_KEYWORDS) and not any(
        has_phrase(lowered, kw) for kw in STUDY_HINT_KEYWORDS
    ):
        return BOUNDARY_REPLY

    wants_answer = any(phrase in lowered for phrase in ASK_FOR_ANSWER_PHRASES)

    if active:
        answer_norm = active["answer"].lower()
        if answer_norm in lowered:
            reply = rng.choice(CORRECT_TEMPLATES).format(answer=active["answer"])
            session_state["active_question"] = None
            return reply
        if wants_answer:
            reply = rng.choice(DIRECT_ANSWER_TEMPLATES).format(answer=active["answer"])
            session_state["active_question"] = None
            return reply
        return rng.choice(HINT_TEMPLATES).format(hint=active["hint"])

    if wants_answer:
        return "I don't have an active practice question yet — try asking me for one first!"

    return rng.choice(GENERIC_GUIDE_TEMPLATES).format(subject=subject)


## Step 5: Optional upgrade — a real OpenAI API key

Leave the field below **blank** to keep using the free rule-based engine
above. If you paste in a real key, Orbit will try to tutor you using
`gpt-4o-mini` with the exact system prompt from Step 1 — and if that call
fails for *any* reason, the notebook quietly falls back to the rule-based
engine instead of crashing.


In [5]:
%pip install -q openai
print("openai package ready (only used if you provide an API key below).")


Note: you may need to restart the kernel to use updated packages.
openai package ready (only used if you provide an API key below).


In [6]:
OPENAI_API_KEY = ""  #@param {type:"string"}


In [7]:
def try_openai_reply(history):
    """Attempt a real OpenAI call. Returns the reply text, or None on any failure."""
    if not OPENAI_API_KEY:
        return None
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=history,
            temperature=0.6,
            max_tokens=350,
        )
        return response.choices[0].message.content
    except Exception as error:
        print(f"(OpenAI call failed, falling back to the local engine: {error})")
        return None


## Step 6: Putting it together — `ask()` and helpers

`ask(user_message)` is the function you'll call directly. It appends your
message to `conversation_history`, tries the optional OpenAI path, falls
back to the rule-based engine, appends the reply, and prints the exchange.


In [8]:
def ask(user_message):
    conversation_history.append({"role": "user", "content": user_message})

    reply = try_openai_reply(conversation_history)
    if reply is None:
        reply = generate_rule_based_reply(user_message, session_state, rng)

    conversation_history.append({"role": "assistant", "content": reply})

    print(f"You: {user_message}")
    print(f"Orbit: {reply}\n")
    return reply


def set_subject_and_level(subject, level):
    """Change the current subject/level (like changing the sidebar pickers)."""
    session_state["subject"] = subject
    session_state["level"] = level
    session_state["active_question"] = None
    conversation_history[0] = {"role": "system", "content": build_system_prompt(subject, level)}
    print(f"Session set to subject={subject!r}, level={level!r}")


## Step 7: Scripted demo — hint-first tutoring in action

This runs automatically with no typing required, so `Runtime → Run all`
always completes. Watch how a wrong first attempt gets a **hint**, not the
answer, and only an explicit "I'm stuck, just tell me" reveals the answer.


In [9]:
set_subject_and_level("Math", "Beginner")

# index=0 makes this deterministic for the demo: "What is 7 + 8?" (hint about 3+5, answer 15)
question = get_practice_question(session_state["subject"], session_state["level"], index=0)
conversation_history.append({"role": "assistant", "content": question})
print(f"Orbit: {question}\n")

ask("Is it 99?")               # a wrong attempt -> should get a hint, not the answer
ask("I'm stuck, just tell me the answer")   # explicit ask -> should get the direct answer


Session set to subject='Math', level='Beginner'
Orbit: What is 7 + 8?

You: Is it 99?
Orbit: Not quite yet — here's a hint: Try breaking 8 into 3 + 5, so it's 7 + 3 + 5.

You: I'm stuck, just tell me the answer
Orbit: Since you've given it a genuine shot, here's the answer: 15.



"Since you've given it a genuine shot, here's the answer: 15."

**Honest note on the rule-based engine:** this default, no-key-required
engine can only tutor on the fixed bank of practice questions in
`PRACTICE_BANK` above, and it checks correctness with simple text matching
rather than truly understanding your reasoning — so it's a much narrower
tutor than a real LLM. What it *can* do reliably is guarantee the
hint-before-answer rule: you can read `generate_rule_based_reply()` above
and see that the direct answer is only ever returned when you explicitly
ask for it, or when you already got it right. If you add a real OpenAI key
in Step 5, Orbit can tutor on *any* topic you ask about, following the same
hint-first system prompt — though (being a general LLM) it is asked, not
mechanically guaranteed, to always hint before answering.


## Step 8: A second example — correct on the first try

Let's also prove the "you got it right" path works, on a different subject.


In [10]:
set_subject_and_level("Science", "Beginner")

# index=0 makes this deterministic: "What gas do plants absorb...?" (answer: carbon dioxide)
question = get_practice_question(session_state["subject"], session_state["level"], index=0)
conversation_history.append({"role": "assistant", "content": question})
print(f"Orbit: {question}\n")

ask("I think it's carbon dioxide")   # matches the stored answer -> should be praised, not hinted


Session set to subject='Science', level='Beginner'
Orbit: What gas do plants absorb from the air to make food?

You: I think it's carbon dioxide
Orbit: You got it! carbon dioxide is right.



'You got it! carbon dioxide is right.'

## Step 9: Try it yourself (optional)

Pick a subject/level, get a practice question, and chat with Orbit. Edit
the cells below and re-run them as many times as you like.


In [11]:
set_subject_and_level("Coding", "Beginner")
question = get_practice_question(session_state["subject"], session_state["level"])
conversation_history.append({"role": "assistant", "content": question})
print(f"Orbit: {question}\n")


Session set to subject='Coding', level='Beginner'
Orbit: In Python, what keyword starts a function definition?



In [12]:
ask("Your answer or question here")


You: Your answer or question here
Orbit: Good attempt! Try this hint: It's a three-letter word, short for 'define'.



"Good attempt! Try this hint: It's a three-letter word, short for 'define'."